# Social Media Sentiment Analysis System
## Academic Data Science and Computing (DSC) Capstone Project
**Notebook:** Exploratory Data Analysis, Feature Extraction & Machine Learning

---
### 📚 DSC Syllabus Mapping Covered in this Notebook:
- **Unit 1:** Data Science Life Cycle (Collection, Cleaning, Feature Extraction, Modeling, Evaluation)
- **Unit 3:** Statistics & Probability (Summary statistics, distributions, Bayes' Theorem)
- **Unit 4 & 5:** Linear Algebra (Document-Term Matrix, TF-IDF vectors, high-dimensional sparsity)
- **Unit 6:** Data Transformation & Integrity (NLP preprocessing, negation safety, leakage prevention)
- **Unit 7:** Optimization (Naive Bayes closed-form estimation vs. Logistic Regression cross-entropy loss)
- **Unit 9:** Time Series Analysis (Daily aggregations, rolling moving averages)
- **Unit 10:** Model Evaluation & Visualization (Confusion matrix, precision, recall, F1-score)

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
sys.path.insert(0, '..')

from src.config import RAW_CSV, PROCESSED_CSV, SENTIMENT_CLASSES, RANDOM_STATE
from src.data_loader import load_dataset, get_dataset_profile
from src.preprocessing import preprocess_dataframe, clean_text
from src.feature_extraction import build_tfidf_vectorizer
from src.train import split_data, train_model
from src.evaluation import compute_metrics, confusion_matrix_df

# Configure plot styles
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries and modules imported successfully.')

## 1. Dataset Loading & Data Profiling (DSC Unit 1 & Unit 6)
We inspect the dataset dimensions, check for missing values, identify duplicates, and analyze class distribution.

In [ ]:
df_raw = load_dataset()
profile = get_dataset_profile(df_raw)

print(f"Total Records : {profile['n_rows']:,}")
print(f"Total Columns : {profile['n_cols']}")
print(f"Duplicate Rows: {profile['duplicates']:,}")
print(f"Missing Values: {profile['missing']}")
print(f"Class Distribution: {profile.get('class_distribution', {})}")
df_raw.head()

## 2. Exploratory Data Analysis & Descriptive Statistics (DSC Unit 3)
Calculating statistical moments (Mean, Median, Standard Deviation, Min, Max) for review word lengths across sentiment classes.

In [ ]:
df_raw['word_count'] = df_raw['text'].apply(lambda x: len(str(x).split()))

stats_by_sentiment = df_raw.groupby('sentiment')['word_count'].agg(['count', 'mean', 'std', 'median', 'min', 'max']).round(2)
print('Descriptive Statistics of Word Length by Sentiment:')
print(stats_by_sentiment)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df_raw, x='sentiment', order=SENTIMENT_CLASSES, ax=axes[0], palette=['#10b981', '#ef4444', '#6366f1'])
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Count')

sns.histplot(data=df_raw, x='word_count', hue='sentiment', kde=True, ax=axes[1], palette=['#10b981', '#ef4444', '#6366f1'])
axes[1].set_title('Review Word Length Distribution')
axes[1].set_xlabel('Word Count')
plt.tight_layout()
plt.show()

## 3. Text Preprocessing Pipeline (DSC Unit 6)
Transforming unstructured natural language text into clean tokens:
1. Lowercasing
2. URL & HTML tag elimination
3. Special character & punctuation removal
4. Tokenization
5. Stopword filtering with **negation safety** (preserving words like 'not', 'never', 'no')

In [ ]:
sample_raw = '<p>I did NOT like this product at all! Check https://example.com #worst</p>'
sample_clean = clean_text(sample_raw)
print('Before Cleaning:', sample_raw)
print('After Cleaning :', sample_clean)

# Preprocess entire DataFrame
df_clean = preprocess_dataframe(df_raw)
print(f'Preprocessed dataset size: {len(df_clean):,} records')
df_clean[['text', 'clean_text', 'sentiment']].head()

## 4. Feature Extraction using TF-IDF (DSC Unit 4: Linear Algebra I)
Text cannot be passed directly into mathematical classifiers. TF-IDF maps each document to a high-dimensional vector in $\mathbb{R}^{V}$ where $V$ is vocabulary size.
$$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \left( \ln\frac{1 + |D|}{1 + |\{d \in D : t \in d\}|} + 1 \right)$$

In [ ]:
X_train, X_test, y_train, y_test = split_data(df_clean)

vectorizer = build_tfidf_vectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f'Vocabulary size (features) : {len(vectorizer.vocabulary_):,}')
print(f'Training Sparse Matrix Shape: {X_train_vec.shape}')
print(f'Sparsity                   : {100.0 * (1.0 - (X_train_vec.nnz / (X_train_vec.shape[0] * X_train_vec.shape[1]))):.2f}%')

## 5. Machine Learning Classifier Training (DSC Unit 3 & Unit 7)
We train **Multinomial Naive Bayes** using Bayes' Theorem with Laplace smoothing:
$$P(c | \mathbf{x}) \propto P(c) \prod_{i=1}^{n} P(w_i | c)$$
And compare against **Logistic Regression** trained with L-BFGS gradient optimization.

In [ ]:
nb_pipe = train_model(X_train, y_train, model_type='naive_bayes')
lr_pipe = train_model(X_train, y_train, model_type='logistic_regression')

y_pred_nb = nb_pipe.predict(X_test)
y_pred_lr = lr_pipe.predict(X_test)

metrics_nb = compute_metrics(y_test, y_pred_nb, labels=SENTIMENT_CLASSES)
metrics_lr = compute_metrics(y_test, y_pred_lr, labels=SENTIMENT_CLASSES)

comp_df = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 Macro', 'Precision Macro', 'Recall Macro'],
    'Multinomial Naive Bayes': [metrics_nb['accuracy'], metrics_nb['f1_macro'], metrics_nb['precision_macro'], metrics_nb['recall_macro']],
    'Logistic Regression': [metrics_lr['accuracy'], metrics_lr['f1_macro'], metrics_lr['precision_macro'], metrics_lr['recall_macro']],
})
print(comp_df.to_string(index=False))

## 6. Confusion Matrix & Detailed Evaluation (DSC Unit 10)
Heatmap visualization of True Positives, False Positives, and False Negatives for each sentiment class.

In [ ]:
cm_df = confusion_matrix_df(metrics_nb)

plt.figure(figsize=(7, 5))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Multinomial Naive Bayes — Confusion Matrix')
plt.ylabel('True Sentiment')
plt.xlabel('Predicted Sentiment')
plt.show()

print('Classification Report:')
print(pd.DataFrame(metrics_nb['classification_report']).transpose())

## 7. Time Series Sentiment Trends (DSC Unit 9)
Aggregating sentiment volume chronologically and analyzing trend lines using 7-day rolling moving averages.

In [ ]:
df_raw['date'] = pd.to_datetime(df_raw['timestamp']).dt.date
daily_counts = df_raw.groupby(['date', 'sentiment']).size().unstack(fill_value=0)

# Compute 7-day Moving Average
daily_ma = daily_counts.rolling(window=7, min_periods=1).mean()

plt.figure(figsize=(12, 5))
for col, color in zip(SENTIMENT_CLASSES, ['#10b981', '#ef4444', '#6366f1']):
    if col in daily_ma.columns:
        plt.plot(daily_ma.index, daily_ma[col], label=f'{col} (7d MA)', color=color, linewidth=2)
plt.title('7-Day Rolling Moving Average Sentiment Trend')
plt.xlabel('Date')
plt.ylabel('Average Daily Reviews')
plt.legend()
plt.show()

## 8. Summary & Capstone Conclusion
- Multinomial Naive Bayes demonstrates high classification performance on sparse TF-IDF vectors.
- Preserving negation tokens during NLP preprocessing is critical to avoid sentiment inversion.
- Time series moving averages effectively smooth out day-to-day noise to reveal underlying opinion trajectories.